# Free Hit - expected points & squad optimisation

Weekly workflow. Run top to bottom; the only cell you normally *edit* is the
minutes-override step.

The model prices every FPL scoring event it can from **live exchange odds** and
falls back to a statistical model where the market cannot reach. Every number
carries a provenance tag so you can see which is which.

## 1. Setup

In [1]:
# Pick up edits to the fplfh package without restarting the kernel. Without
# this, a running kernel keeps the version of a module it first imported, and
# any function added since fails to import until you restart.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass          # not running under IPython

import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "fplfh").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from fplfh.pipeline import run
from fplfh.optimise import optimise_free_hit, top_squads
from fplfh.scoring import validate_against_api
from fplfh.fpl import FPLClient
from fplfh.model import COMPONENTS

## 2. Confirm the scoring rules

The points-per-event table is read live from the FPL API, but the *thresholds*
(DefCon 10/12, saves per 3, conceded per 2) are not published and were derived by
boundary search over finished gameweeks. This re-runs that check against the
latest results - if FPL changes a rule mid-season, it shows up here rather than
silently skewing every projection.

In [2]:
report = validate_against_api(FPLClient())
print("rules consistent with live data:", report["ok"])
for name, finding in report["findings"].items():
    print(f"  {name:<20} {finding}")

rules consistent with live data: True
  defcon_DEF           {'config_threshold': 10, 'min_scoring': 10, 'max_non_scoring': 9}
  defcon_MID           {'config_threshold': 12, 'min_scoring': 12, 'max_non_scoring': 11}
  defcon_FWD           {'config_threshold': 12, 'min_scoring': None, 'max_non_scoring': 8, 'note': 'no FWD reached the threshold in the scanned gameweeks'}
  saves_divisor        {'config_per_n': 3, 'min_saves_scoring': 3, 'consistent': True}


## 3. Run the pipeline

Set `odds_source="none"` to force the fallback model (useful for comparing what
the market is actually adding). With an API key in `.env`, fixtures with usable
odds are priced from the market and the rest fall back automatically.

Every API call is cached (bootstrap/fixtures/live stats 1hr, odds 15min), so
re-running this notebook normally costs nothing. Set `REFRESH = True` below to
force a genuine refresh of all of it - prices, injury news, live minutes, and
market odds - right before you act on the output. It burns a small amount of
Odds API quota (2 credits/fixture) each time, so leave it `False` for routine
re-runs.

In [3]:
REFRESH = False                 # True = force-refresh prices, news, odds (costs quota)

res = run(event=None, max_age=0 if REFRESH else None)   # event=None -> next gameweek

print(f"\nGameweek {res.event}")
print(f"Market share of total xP: {100*res.exchange_share():.1f}%")
res.provenance

Odds API: priced 10/10 fixtures  [quota unknown (no live call yet this session)]
GW5: 659 players, 10 fixtures

Gameweek 5
Market share of total xP: 100.0%


,scoreline_source,player_fixtures,share_of_total_xp
0,oddsapi:betfair_ex_uk:h2h+totals(3),60,0.10
1,oddsapi:betfair_ex_uk:h2h+totals(4),127,0.20
2,oddsapi:betfair_ex_uk:h2h+totals(5),472,0.70
3,[goals] model:xg_share,659,1.00


### Fixture view

`xG_home`/`xG_away` are the fitted Dixon-Coles goal rates. Clean-sheet and
goals-conceded expectations are read off the same joint distribution, so they
cannot contradict the match odds.

In [4]:
res.fixture_table()

,fixture,kickoff,xG_home,xG_away,P(home),P(draw),P(away),CS_home,CS_away,source
0,BRE v CHE,2026-09-18 19:00:00+00:00,1.59,1.73,0.34,0.25,0.40,0.18,0.20,oddsapi:betfair_ex_uk:h2h+totals(3)
1,TOT v AVL,2026-09-19 11:30:00+00:00,1.66,1.15,0.48,0.27,0.25,0.32,0.19,oddsapi:betfair_ex_uk:h2h+totals(5)
2,BHA v ARS,2026-09-19 14:00:00+00:00,0.99,1.84,0.19,0.24,0.56,0.16,0.37,oddsapi:betfair_ex_uk:h2h+totals(5)
3,EVE v IPS,2026-09-19 14:00:00+00:00,1.83,1.07,0.54,0.25,0.21,0.34,0.16,oddsapi:betfair_ex_uk:h2h+totals(5)
4,NEW v HUL,2026-09-19 14:00:00+00:00,2.02,1.04,0.60,0.22,0.19,0.35,0.13,oddsapi:betfair_ex_uk:h2h+totals(5)
5,NFO v COV,2026-09-19 16:30:00+00:00,1.91,0.95,0.59,0.24,0.17,0.39,0.15,oddsapi:betfair_ex_uk:h2h+totals(5)
6,BOU v LIV,2026-09-20 13:00:00+00:00,1.50,1.84,0.30,0.25,0.45,0.16,0.22,oddsapi:betfair_ex_uk:h2h+totals(4)
7,LEE v CRY,2026-09-20 13:00:00+00:00,1.79,1.04,0.54,0.25,0.21,0.35,0.17,oddsapi:betfair_ex_uk:h2h+totals(5)
8,MCI v SUN,2026-09-20 13:00:00+00:00,2.43,0.69,0.76,0.16,0.09,0.50,0.09,oddsapi:betfair_ex_uk:h2h+totals(5)
9,FUL v MUN,2026-09-20 15:30:00+00:00,1.35,1.80,0.28,0.25,0.47,0.17,0.26,oddsapi:betfair_ex_uk:h2h+totals(4)


## 4. Minutes - the part worth your judgement

The auto model reads FPL availability flags and recent starts. It cannot hear a
press conference, so review the players below and override anything you disagree
with in **`config/minutes_overrides.yaml`**, then re-run section 3.

Shorthand is just expected minutes:

```yaml
players:
  Haaland: 0      # ruled out
  Saka: 60        # expected to be managed
```

In [5]:
m = res.minutes
flagged = m[(m.status != "a") & (m.price >= 4.5)].sort_values("price", ascending=False)
print("FLAGGED PLAYERS (auto-downgraded):")
flagged[["web_name","team","position","price","status","chance_of_playing",
         "p_start","p_60","xmins","news"]].head(20)

FLAGGED PLAYERS (auto-downgraded):


,web_name,team,position,price,status,chance_of_playing,p_start,p_60,xmins,news
51,Watkins,AVL,FWD,7.80,u,0.00,0.00,0.00,0.00,Has joined Al Hilal permanently
181,João Pedro,CHE,FWD,7.80,d,75.00,0.69,0.67,60.43,Unspecified injury - 75% chance of playing
85,Kroupi.Jr,BOU,MID,7.40,i,0.00,0.00,0.00,0.00,Foot injury - Expected back 7 Nov
456,Ekitiké,LIV,FWD,7.40,i,0.00,0.00,0.00,0.00,Achilles injury - Unknown return date
479,Doku,MCI,MID,7.40,i,0.00,0.00,0.00,0.00,Calf injury - Expected back 20 Sep
477,Foden,MCI,MID,7.00,s,0.00,0.00,0.00,0.00,Suspended until 17 Oct
480,Rodrigo,MCI,MID,6.50,u,0.00,0.00,0.00,0.00,Has joined Barcelona permanently
619,Kulusevski,TOT,MID,6.50,i,0.00,0.00,0.00,0.00,Knee injury - Unknown return date
251,Mateta,CRY,FWD,6.30,i,0.00,0.00,0.00,0.00,Hamstring injury - Expected back 11 Oct
17,Martinelli,ARS,MID,6.30,u,0.00,0.00,0.00,0.00,Has joined Al Hilal permanently


In [6]:
# Rotation risk: players the model is unsure about who still make the squad shortlist
risky = res.players.merge(m[["player_id","p_start","status"]], on="player_id", suffixes=("","_m"))
risky = risky[(risky.xp > 2.0) & (risky.p_start_m.between(0.25, 0.75))]
print("ROTATION RISKS among viable picks - worth a manual call:")
risky.sort_values("xp", ascending=False)[
    ["web_name","team","position","price","opponent","p_start_m","xmins","xp"]].head(15)

ROTATION RISKS among viable picks - worth a manual call:


,web_name,team,position,price,opponent,p_start_m,xmins,xp
59,João Pedro,CHE,FWD,7.80,BRE,0.69,60.43,3.85
77,Elvedi,LEE,DEF,4.50,CRY,0.62,56.37,3.57
82,Gakpo,LIV,MID,7.20,BOU,0.63,56.32,3.50
84,Sávio,TOT,MID,6.50,AVL,0.65,57.65,3.48
87,Barcola,LIV,MID,8.00,BOU,0.57,51.01,3.46
94,Enzo,MCI,MID,6.90,SUN,0.60,53.98,3.27
97,Delap,NFO,FWD,5.50,COV,0.65,58.05,3.25
103,Johnson,EVE,MID,5.90,IPS,0.62,56.80,3.17
106,Ndiaye,MCI,MID,5.90,SUN,0.61,54.99,3.12
110,Willock,NEW,MID,5.00,HUL,0.55,51.41,3.08


In [7]:
# Did every override actually apply? A typo here otherwise does nothing silently.
unmatched = [w for w in res.warnings if "override not matched" in w]
print("unmatched overrides:", unmatched or "none - all applied")
overridden = m[m.minutes_source == "override"]
print(f"{len(overridden)} manual override(s) in effect")
overridden[["web_name","team","position","p_start","p_60","xmins"]]

unmatched overrides: none - all applied
0 manual override(s) in effect


,web_name,team,position,p_start,p_60,xmins


## 5. Expected points

`xp` is the sum of the component columns. Reading the components is usually more
informative than the total - it shows *why* a player rates, and therefore which
assumption to challenge.

In [8]:
cols = ["web_name","team","position","price","opponent","is_home"] + COMPONENTS + ["xp"]
res.players.head(30)[cols]

,web_name,team,position,price,opponent,is_home,xp_minutes,xp_goals,xp_assists,xp_clean_sheet,xp_conceded,xp_defcon,xp_saves,xp_cards,xp_bonus,xp_pens,xp
0,Haaland,MCI,FWD,15.50,SUN,H,1.78,4.20,0.47,0.00,0.00,0.01,0.00,-0.09,1.55,-0.08,7.83
1,Gibbs-White,NFO,MID,8.00,COV,H,1.94,2.30,1.26,0.38,0.00,0.02,0.00,-0.15,0.93,-0.07,6.62
2,Saka,ARS,MID,9.50,BHA,A,1.88,2.71,0.73,0.36,0.00,0.16,0.00,-0.14,0.96,-0.06,6.59
3,Barry,EVE,FWD,5.60,IPS,H,1.87,3.32,0.22,0.00,0.00,0.00,0.00,-0.11,1.21,-0.06,6.45
4,B.Fernandes,MUN,MID,12.00,FUL,A,1.91,2.29,1.08,0.26,0.00,0.05,0.00,-0.15,0.90,-0.06,6.27
5,Wissa,NEW,FWD,6.20,HUL,H,1.94,2.84,0.14,0.00,0.00,0.02,0.00,-0.16,1.03,0.00,5.81
6,Calvert-Lewin,LEE,FWD,6.00,CRY,H,1.89,2.78,0.17,0.00,0.00,0.00,0.00,-0.18,1.01,-0.06,5.61
7,Guéhi,MCI,DEF,6.00,SUN,H,1.83,1.15,0.40,1.83,-0.14,0.17,0.00,-0.14,0.49,0.00,5.61
8,Mbeumo,MUN,MID,7.90,FUL,A,1.89,2.21,0.59,0.26,0.00,0.04,0.00,-0.17,0.77,0.00,5.59
9,Isak,LIV,FWD,9.10,BOU,A,1.77,2.73,0.20,0.00,0.00,0.00,0.00,-0.10,0.99,0.00,5.59


In [9]:
# Best value per million - useful for filling the cheap end of the squad
v = res.players[res.players.xp > 1.0].copy()
v["xp_per_m"] = v.xp / v.price
v.sort_values("xp_per_m", ascending=False).head(20)[
    ["web_name","team","position","price","opponent","xp","xp_per_m"]]

,web_name,team,position,price,opponent,xp,xp_per_m
11,Bogle,LEE,DEF,4.50,CRY,5.29,1.18
3,Barry,EVE,FWD,5.60,IPS,6.45,1.15
27,Justin,LEE,DEF,4.50,CRY,4.50,1.00
16,Hall,NEW,DEF,5.20,HUL,5.15,0.99
23,Muharemović,LEE,DEF,5.00,CRY,4.75,0.95
5,Wissa,NEW,FWD,6.20,HUL,5.81,0.94
6,Calvert-Lewin,LEE,FWD,6.00,CRY,5.61,0.94
7,Guéhi,MCI,DEF,6.00,SUN,5.61,0.94
17,Murillo,NFO,DEF,5.50,COV,5.13,0.93
68,Thomas,COV,DEF,4.00,NFO,3.69,0.92


In [10]:
# Defensive contribution leaders - the newest scoring route, and the one the
# market cannot price directly. FWD DefCon is worth ~0.02 pts/game: ignore it.
d = res.player_fixtures
d[d.xp_defcon > 0].sort_values("xp_defcon", ascending=False).head(15)[
    ["web_name","team","position","price","opponent","dc90","p_defcon","xp_defcon","xp"]]

,web_name,team,position,price,opponent,dc90,p_defcon,xp_defcon,xp
276,Egan,HUL,DEF,4.10,NEW,13.75,0.69,1.39,3.37
532,Mukiele,SUN,DEF,5.40,MCI,13.00,0.61,1.22,3.01
333,Muharemović,LEE,DEF,5.00,CRY,12.50,0.60,1.19,4.75
587,Khalaili,CRY,DEF,5.00,LEE,11.83,0.53,1.06,4.17
97,Janelt,BRE,MID,5.00,CHE,13.75,0.52,1.04,3.88
201,Richards,CRY,DEF,5.00,LEE,11.00,0.51,1.03,3.27
618,Palacios,IPS,MID,5.00,EVE,17.54,0.50,1.00,2.76
68,Scott,BOU,MID,6.10,LIV,13.15,0.48,0.95,4.30
144,Fofana,CHE,DEF,5.00,BRE,11.90,0.46,0.93,3.09
431,Mainoo,MUN,MID,5.50,FUL,14.40,0.45,0.91,3.78


## 6. Optimise the squad

Exact MILP solve: 15 players, 2/5/5/3, max 3 per club, valid XI, captain doubled,
bench discounted to its autosub value.

On a real Free Hit your budget is **current squad value + bank**, not 100.0 -
set it below.

In [11]:
BUDGET = 100.0          # <-- your squad value + bank
LOCK   = []             # e.g. ["Haaland"] to force in
BAN    = []             # e.g. ["Saka"] to force out

squad = optimise_free_hit(res.players, budget=BUDGET, locked=LOCK, banned=BAN)
print(squad.summary())

Formation 3-4-3   cost L99.9m   XI xP 72.00 (incl. captain)
Captain: Haaland   Vice: Gibbs-White

  GKP  Donnarumma         MCI  L 5.5  SUN      xP  3.96
  DEF  Guéhi              MCI  L 6.0  SUN      xP  5.61
  DEF  Bogle              LEE  L 4.5  CRY      xP  5.29
  DEF  Hall               NEW  L 5.2  HUL      xP  5.15
  MID  Gibbs-White        NFO  L 8.0  COV      xP  6.62
  MID  Saka               ARS  L 9.5  BHA      xP  6.59
  MID  Mbeumo             MUN  L 7.9  FUL      xP  5.59
  MID  Stach              LEE  L 6.0  CRY      xP  5.25
  FWD  Haaland            MCI  L15.5  SUN      xP  7.83 (C)
  FWD  Barry              EVE  L 5.6  IPS      xP  6.45
  FWD  Wissa              NEW  L 6.2  HUL      xP  5.81

  Bench:
  GKP  Kinsky             TOT  L 4.5  AVL      xP  3.25
  DEF  Murillo            NFO  L 5.5  COV      xP  5.13
  DEF  Justin             LEE  L 4.5  CRY      xP  4.50
  MID  Ndoye              NFO  L 5.5  COV      xP  4.84


In [12]:
# Full squad detail with component breakdown
squad.players[["web_name","team","position","price","opponent","is_starter","is_captain"]
              + COMPONENTS + ["xp"]]

,web_name,team,position,price,opponent,is_starter,is_captain,xp_minutes,xp_goals,xp_assists,xp_clean_sheet,xp_conceded,xp_defcon,xp_saves,xp_cards,xp_bonus,xp_pens,xp
46,Donnarumma,MCI,GKP,5.50,SUN,True,False,1.81,0.00,0.00,1.81,-0.13,0.00,0.29,-0.05,0.24,0.00,3.96
98,Kinsky,TOT,GKP,4.50,AVL,False,False,1.83,0.00,0.00,1.19,-0.30,0.00,0.42,-0.06,0.17,0.00,3.25
7,Guéhi,MCI,DEF,6.00,SUN,True,False,1.83,1.15,0.40,1.83,-0.14,0.17,0.00,-0.14,0.49,0.00,5.61
11,Bogle,LEE,DEF,4.50,CRY,True,False,1.89,1.70,0.23,1.37,-0.27,0.07,0.00,-0.18,0.48,0.00,5.29
16,Hall,NEW,DEF,5.20,HUL,True,False,1.94,0.24,0.96,1.41,-0.28,0.61,0.00,-0.15,0.41,0.00,5.15
17,Murillo,NFO,DEF,5.50,COV,False,False,1.94,0.49,0.34,1.54,-0.24,0.86,0.00,-0.18,0.39,0.00,5.13
27,Justin,LEE,DEF,4.50,CRY,False,False,1.89,0.30,0.49,1.37,-0.27,0.54,0.00,-0.15,0.33,0.00,4.50
1,Gibbs-White,NFO,MID,8.00,COV,True,False,1.94,2.30,1.26,0.38,0.00,0.02,0.00,-0.15,0.93,-0.07,6.62
2,Saka,ARS,MID,9.50,BHA,True,False,1.88,2.71,0.73,0.36,0.00,0.16,0.00,-0.14,0.96,-0.06,6.59
8,Mbeumo,MUN,MID,7.90,FUL,True,False,1.89,2.21,0.59,0.26,0.00,0.04,0.00,-0.17,0.77,0.00,5.59


### Alternatives

The optimum is often only a fraction of a point clear of quite different squads.
Each alternative differs by at least `diversity` places in the starting XI.

In [13]:
alts = top_squads(res.players, n=4, budget=BUDGET, locked=LOCK, banned=BAN, diversity=3)
for i, s in enumerate(alts, 1):
    print(f"{i}. {s.formation}  XI xP {s.starting_xp:5.2f}  cost {s.cost:5.1f}m  "
          f"C={s.captain:<14} bench={', '.join(s.bench)}")

1. 3-4-3  XI xP 72.00  cost  99.9m  C=Haaland        bench=Murillo, Ndoye, Justin, Kinsky
2. 3-4-3  XI xP 71.73  cost 100.0m  C=Haaland        bench=Stach, Hall, Justin, Kinsky
3. 3-4-3  XI xP 72.32  cost  99.9m  C=Haaland        bench=Thomas, Egan, Slater, Bentley
4. 3-4-3  XI xP 71.81  cost  99.7m  C=Haaland        bench=Ndoye, Justin, Thomas, Bentley


In [14]:
# Captaincy: the armband is worth a full extra return, so check the top few
res.players.head(8)[["web_name","team","position","price","opponent","xp_goals","xp_assists","xp"]]

,web_name,team,position,price,opponent,xp_goals,xp_assists,xp
0,Haaland,MCI,FWD,15.50,SUN,4.20,0.47,7.83
1,Gibbs-White,NFO,MID,8.00,COV,2.30,1.26,6.62
2,Saka,ARS,MID,9.50,BHA,2.71,0.73,6.59
3,Barry,EVE,FWD,5.60,IPS,3.32,0.22,6.45
4,B.Fernandes,MUN,MID,12.00,FUL,2.29,1.08,6.27
5,Wissa,NEW,FWD,6.20,HUL,2.84,0.14,5.81
6,Calvert-Lewin,LEE,FWD,6.00,CRY,2.78,0.17,5.61
7,Guéhi,MCI,DEF,6.00,SUN,1.15,0.40,5.61


## 7. How much is the market actually adding?

Run both ways and compare. If the two agree closely, the market is confirming
the model; where they diverge, the market is usually the better estimate - it has
information (team news, money) the statistical model does not.

In [15]:
res_model = run(odds_source="none", verbose=False)
cmp = res.players[["player_id","web_name","team","position","price","xp"]].merge(
    res_model.players[["player_id","xp"]], on="player_id", suffixes=("_market","_model"))
cmp["diff"] = cmp.xp_market - cmp.xp_model
if cmp["diff"].abs().sum() < 1e-9:
    print("Identical - no market data in use (no API key, or no markets matched).")
else:
    print("Largest disagreements (market vs fallback model):")
    display(cmp.reindex(cmp["diff"].abs().sort_values(ascending=False).index).head(15))

Largest disagreements (market vs fallback model):


,player_id,web_name,team,position,price,xp_market,xp_model,diff
33,8,Calafiori,ARS,DEF,5.80,4.38,3.03,1.35
112,115,De Cuyper,BHA,DEF,4.80,3.04,4.39,-1.35
0,411,Haaland,MCI,FWD,15.50,7.83,6.57,1.27
7,388,Guéhi,MCI,DEF,6.00,5.61,4.36,1.25
85,124,Groß,BHA,MID,5.70,3.47,4.69,-1.22
38,4,Gabriel,ARS,DEF,8.00,4.18,3.01,1.17
72,542,E.Le Fée,SUN,MID,5.80,3.66,4.83,-1.17
9,379,Isak,LIV,FWD,9.10,5.59,4.50,1.09
24,390,Rúben,MCI,DEF,5.50,4.72,3.63,1.09
5,464,Wissa,NEW,FWD,6.20,5.81,4.74,1.07


## 8. Export

In [16]:
from fplfh.config import OUT_DIR, ensure_dirs
ensure_dirs()
res.players.to_csv(OUT_DIR / f"xp_gw{res.event}.csv", index=False)
squad.players.to_csv(OUT_DIR / f"squad_gw{res.event}.csv", index=False)
res.fixture_table().to_csv(OUT_DIR / f"fixtures_gw{res.event}.csv", index=False)
print("written to", OUT_DIR)

written to C:\Users\alexg\OneDrive\Desktop\Projects\Fantasy-PL\data\out
